# Atelier 3.D — Observabilité et débogage ★

**Durée : 40 min** · Projet fil rouge, incrément du Jour 3

Un agent décide lui-même de ses étapes : sans **trace**, impossible de savoir pourquoi il a réussi ou échoué. Vous instrumentez l'agent du service après-vente, puis vous diagnostiquez trois pannes classiques :

1. un outil qui plante ;
2. des outils mal décrits ;
3. une boucle sans fin.

Deux traceurs : un **traceur local** (`af.TraceurLocal`, sans service externe) et **Langfuse** si ses clés sont renseignées dans `.env`.

In [1]:
import os

import pandas as pd
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.errors import GraphRecursionError
from langgraph.types import Command

import agents_fictiva as af
import crm
import outils_llm as llm

MODELE = llm.choisir_modele()
outils_mcp = await af.outils_crm_mcp()
print("Modèle :", MODELE, "| Langfuse :", "actif" if af.langfuse_actif() else "non configuré")

def formater(demande):
    return f"De : {demande['expediteur']} ({demande['entreprise']})\nObjet : {demande['objet']}\n\n{demande['corps']}"


async def executer(agent, texte, thread, limite=25):
    """Exécute l'agent avec un traceur local (et Langfuse s'il est configuré) ; valide les tickets."""
    traceur = af.TraceurLocal()
    rappels = [traceur] + ([af.traceur_langfuse()] if af.langfuse_actif() else [])
    configuration = af.config(thread, *rappels, limite=limite)
    resultat = await agent.ainvoke({"messages": [HumanMessage(texte)]}, configuration)
    while resultat.get("__interrupt__"):
        resultat = await agent.ainvoke(Command(resume="oui"), configuration)
    return resultat, traceur

Modèle : nvidia:nvidia/nemotron-3.5-lightning-30b-a3b | Langfuse : non configuré


## 1. Lire une trace

Chaque ligne est un appel de modèle ou d'outil : ce que le modèle a décidé, ce que l'outil a renvoyé, les tokens consommés, la durée.

In [2]:
agent = af.construire_agent(outils_mcp + [af.calculer_remboursement, af.creer_ticket], modele=MODELE)
demandes = af.charger_demandes()
resultat, traceur = await executer(agent, formater(demandes[1]), "trace-1")
traceur.tableau()

,type,nom,entree,sortie,tokens_entree,tokens_sortie,duree_s,erreur
0,modèle,LLM,2 messages,→ rechercher_client,1384.0,323.0,2.99,None
1,outil,rechercher_client,{'numero_client': 'CL-10012'},"{\n ""numero_client"": ""CL-10012"",\n ""nom"": ""R...",NaN,NaN,0.66,None
2,modèle,LLM,4 messages,→ commandes_du_client,1488.0,58.0,0.60,None
3,outil,commandes_du_client,{'numero_client': 'CL-10012'},"{\n ""numero_client"": ""CL-10012"",\n ""commande...",NaN,NaN,0.89,None
4,modèle,LLM,6 messages,→ consulter_commande,1696.0,136.0,1.42,None
5,outil,consulter_commande,{'numero_commande': 'CMD-2026-0331'},"{\n ""numero_commande"": ""CMD-2026-0331"",\n ""c...",NaN,NaN,0.67,None
6,modèle,LLM,8 messages,→ calculer_remboursement,2005.0,290.0,2.94,None
7,outil,calculer_remboursement,"{'consommables_ouverts': False, 'references': ...","{""remboursable"": true, ""montant_chf"": 360.0, ""...",NaN,NaN,0.00,None
8,modèle,LLM,10 messages,→ creer_ticket,2185.0,210.0,1.64,None
9,outil,creer_ticket,"{'numero_client': 'CL-10012', 'resume': 'Clien...",en attente de validation humaine,NaN,NaN,0.00,None


In [3]:
traceur.bilan()

{'appels_modele': 7,
 'appels_outils': 8,
 'erreurs': 0,
 'tokens_entree': 14283,
 'tokens_sortie': 2834,
 'duree_totale_s': 26.3}

**Langfuse.** Avec les clés de l'offre gratuite de Langfuse Cloud (ou d'une instance auto-hébergée) dans `.env`, la même exécution apparaît dans l'interface web de Langfuse : arbre des étapes, entrées et sorties, tokens, coûts, durées. Le branchement tient en une ligne : `callbacks=[CallbackHandler()]`.

In [4]:
if af.langfuse_actif():
    print("Trace envoyée à Langfuse :", os.getenv("LANGFUSE_BASE_URL", "https://cloud.langfuse.com"))
else:
    print("Langfuse non configuré : renseignez LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY et LANGFUSE_BASE_URL dans .env.")

Langfuse non configuré : renseignez LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY et LANGFUSE_BASE_URL dans .env.


## 2. Panne n° 1 : un outil qui plante

Une première version de l'outil de consultation, écrite trop vite : elle suppose que le numéro de commande existe. Le client s'est trompé de numéro : `CMD-2026-0451` au lieu de `CMD-2026-0415`.

In [5]:
@tool
def consulter_commande(numero_commande: str) -> dict:
    """Détail d'une commande (format CMD-AAAA-NNNN)."""
    return crm.DONNEES["commandes"][numero_commande]          # KeyError si le numéro n'existe pas

texte = formater(demandes[0]).replace("CMD-2026-0415", "CMD-2026-0451")
fragile = af.construire_agent([consulter_commande, af.calculer_remboursement], modele=MODELE, erreurs_outils=False)
try:
    resultat, traceur = await executer(fragile, texte, "panne-1")
    print("L'agent a terminé sans erreur :", resultat["messages"][-1].content[:300])
except Exception as erreur:
    print("L'agent s'arrête :", type(erreur).__name__, erreur)

L'agent s'arrête : KeyError 'CMD-2026-0451'


**Diagnostic.** L'exception remonte et interrompt tout le traitement : le client n'aura pas de réponse. Deux corrections, complémentaires :
1. le graphe renvoie l'erreur au modèle au lieu de s'arrêter (`erreurs_outils=True`) ;
2. surtout, l'outil renvoie un **message d'erreur exploitable**, qui dit au modèle quoi faire (c'est le cas de `crm.consulter_commande`).

In [6]:
robuste = af.construire_agent(outils_mcp + [af.calculer_remboursement], modele=MODELE, erreurs_outils=True)
resultat, traceur = await executer(robuste, texte, "panne-1-corrigee")
print(traceur.tableau()[["type", "nom", "entree", "sortie", "erreur"]].to_string())
print("\n" + resultat["messages"][-1].content)

      type                     nom                                                                                                             entree                                                                                                                         sortie erreur
0   modèle                     LLM                                                                                                         2 messages                                                                                                            → rechercher_client   None
1    outil       rechercher_client                                                                                      {'numero_client': 'CL-10456'}  {\n  "numero_client": "CL-10456",\n  "nom": "Chevalier Conseil",\n  "contact": "Émilie Chevalier",\n  "statut": "standard",\n   None
2   modèle                     LLM                                                                                                         4 message

## 3. Panne n° 2 : des outils mal décrits

Mêmes fonctions, mais des noms, des descriptions et des paramètres vagues (`code`, `liste`, `option`). Le modèle doit deviner quel outil fait quoi.

In [7]:
@tool
def outil_a(code: str) -> dict:
    """Outil du CRM."""
    return crm.commandes_du_client(code)

@tool
def outil_b(code: str) -> dict:
    """Outil du CRM."""
    return crm.consulter_commande(code)

@tool
def outil_c(code: str, liste: list[str], option: bool, option2: bool = False) -> dict:
    """Outil du CRM."""
    return crm.calculer_remboursement(code, liste, option, option2)

flou = af.construire_agent([outil_a, outil_b, outil_c], modele=MODELE)
precis = af.construire_agent(outils_mcp + [af.calculer_remboursement], modele=MODELE)

comparaison = {}
for nom, agent_test in [("outils mal décrits", flou), ("outils bien décrits", precis)]:
    resultat, traceur = await executer(agent_test, formater(demandes[1]), f"panne-2-{nom}")
    comparaison[nom] = traceur.bilan()
    print(f"=== {nom}\n", traceur.tableau()[["type", "nom", "sortie"]].to_string(), "\n")
pd.DataFrame(comparaison)

=== outils mal décrits
       type      nom                                                                                                                      sortie
0   modèle      LLM                                                                                                                   → outil_a
1    outil  outil_a    {"numero_client": "CL-10012", "commandes": [{"numero_commande": "CMD-2026-0331", "date_commande": "2026-09-04", "statut"
2   modèle      LLM                                                                                                                   → outil_a
3    outil  outil_a                                                  {"erreur": "Client CMD-2026-0331 introuvable. Format attendu : CL-xxxxx."}
4   modèle      LLM                                                                                                                   → outil_b
5    outil  outil_b                                                {"erreur": "Commande CL-10012 introuvable. Fo

=== outils bien décrits
      type                     nom                                                                                                                         sortie
0  modèle                     LLM                                                                                                          → commandes_du_client
1   outil     commandes_du_client  {\n  "numero_client": "CL-10012",\n  "commandes": [\n    {\n      "numero_commande": "CMD-2026-0331",\n      "date_commande":
2  modèle                     LLM                                                                                                           → consulter_commande
3   outil      consulter_commande  {\n  "numero_commande": "CMD-2026-0331",\n  "client": {\n    "numero_client": "CL-10012",\n    "nom": "Roux Avocats",\n    "s
4  modèle                     LLM                                                                                                       → calculer_remboursement
5   outil

,outils mal décrits,outils bien décrits
appels_modele,10.0,4.0
appels_outils,9.0,3.0
erreurs,0.0,0.0
tokens_entree,14378.0,6079.0
tokens_sortie,3860.0,1252.0
duree_totale_s,44.9,11.2


**Diagnostic.** Avec des outils mal décrits, l'agent tâtonne : appels inutiles ou dans le désordre, arguments mal choisis, parfois une réponse sans calcul. Le nom et la description d'un outil font partie du prompt.

## 4. Panne n° 3 : la boucle sans fin

Un outil de stock qui répond « vérification en cours, rappelez-moi » : l'agent obéit, encore et encore.

In [8]:
@tool
def verifier_stock(reference: str) -> dict:
    """Vérifie le stock disponible d'une référence."""
    return {"statut": "en_cours", "message": "Vérification en cours : rappelez verifier_stock pour obtenir le résultat."}

PROMPT_STOCK = ("Tu réponds aux questions de stock des clients de Fictiva. Vérifie toujours le stock avec l'outil "
                "et ne réponds pas avant d'avoir obtenu le résultat de la vérification.")
boucle = af.construire_agent([verifier_stock], prompt=PROMPT_STOCK, modele=MODELE)
traceur = af.TraceurLocal()
try:
    resultat = await boucle.ainvoke({"messages": [HumanMessage("Avez-vous 15 portables NB-14 en stock ?")]},
                                    af.config("panne-3", traceur, limite=12))
    print("Réponse :", resultat["messages"][-1].content)
except GraphRecursionError:
    print("⛔ Limite d'étapes atteinte (recursion_limit = 12) : l'agent tournait en rond.")
print("Appels de verifier_stock :", int((traceur.tableau().nom == "verifier_stock").sum()))

Réponse : Vérification en cours : le système indique que la vérification du stock pour la référence **NB‑14** est toujours en cours. Dès que le résultat sera disponible, je vous le communiquerai.
Appels de verifier_stock : 4


**Diagnostic et correction.** Selon le modèle, l'agent appelle l'outil en boucle jusqu'à la limite d'étapes (`recursion_limit`), ou s'arrête de lui-même après quelques appels avec une réponse creuse (« je vous reviens dès que possible »), qu'il ne pourra jamais tenir. La limite est un filet de sécurité, pas une solution : la cause est la **conception de l'outil**, qui ne doit jamais demander à l'agent de l'interroger en boucle. Corrigeons l'outil et le prompt.

In [9]:
@tool
def verifier_stock(reference: str) -> dict:
    """Vérifie le stock disponible d'une référence. Si le service est indisponible, ne pas réessayer :
    informer le client que la disponibilité lui sera confirmée par email."""
    return {"statut": "indisponible", "message": "Service de stock momentanément indisponible ; ne pas réessayer."}

PROMPT_STOCK = ("Tu réponds aux questions de stock des clients de Fictiva. Vérifie le stock avec l'outil. "
                "N'appelle jamais deux fois un outil avec les mêmes arguments.")
corrige = af.construire_agent([verifier_stock], prompt=PROMPT_STOCK, modele=MODELE)
resultat, traceur = await executer(corrige, "Avez-vous 15 portables NB-14 en stock ?", "panne-3-corrigee", limite=12)
print(traceur.bilan())
print(resultat["messages"][-1].content)

{'appels_modele': 2, 'appels_outils': 1, 'erreurs': 0, 'tokens_entree': 779, 'tokens_sortie': 247, 'duree_totale_s': 2.3}
Le service de vérification du stock est momentanément indisponible. Votre demande de disponibilité pour 15 portables NB‑14 sera confirmée par email dès que le service sera à nouveau disponible.


## 5. Méthode de diagnostic

| Symptôme dans la trace | Cause probable | Correction |
|---|---|---|
| Exception, traitement interrompu | Outil fragile | Messages d'erreur exploitables, erreurs renvoyées au modèle |
| Appels inutiles, mauvais outil, mauvais arguments | Noms ou descriptions vagues | Réécrire noms, descriptions et schémas |
| Même outil appelé en boucle | Outil qui invite à réessayer, prompt sans règle d'arrêt | Résultat définitif, règle d'arrêt, limite d'étapes |
| Réponse sans appel d'outil | Prompt flou, outil non adapté | Rendre l'usage de l'outil explicite dans la méthode |
| Tokens en hausse à chaque tour | Contexte qui s'accumule | Résumer, tronquer les résultats d'outils trop longs |

## À retenir

- Pas d'agent en production sans **trace** de chaque appel de modèle et d'outil.
- La plupart des pannes d'agents viennent des **outils** (robustesse, descriptions, sémantique des erreurs), pas du modèle.
- Les limites (itérations, étapes, budget) sont des filets de sécurité ; la correction se fait dans la conception.
- Langfuse, LangSmith ou Arize Phoenix ajoutent l'historique, la recherche de traces, les coûts et l'évaluation continue.